In [ ]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

PROCESSING=False 
PROCESSING=True #set to True if using Turbulence-Removed Binary Arrays
Processed_string = GetProcessedString(PROCESSING=PROCESSING)

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#IMPORT FUNCTIONS
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation",
                                dtype='int32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Load Data Functions
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z', 'Y', 'X']
    dataDictionary = MakeDataDictionary(variableNames,t)
    Z,Y,X = (dataDictionary[k] for k in variableNames)
    return Z,Y,X

def GetVariableData(t):    
    variableNames = ['QV']
    dataDictionary = MakeDataDictionary(variableNames,t)
    QV, = (dataDictionary[k] for k in variableNames)
    return QV
def GetAData(t):  
    variableNames = [f'{Processed_string}A_g', f'{Processed_string}A_c']

    dataDictionary = MakeDataDictionary(variableNames,t, printstatement=False)
    A_g,A_c = (dataDictionary[k] for k in variableNames)
    return A_g,A_c

def GetAData_Eulerian(t):
    variableNames = [f'A_g', f'A_c']
    [A_g,A_c] = [CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName) for variableName in variableNames]
    return A_g,A_c
def GetVariableData_Eulerian(t):
    variableNames = [f'qv']
    [QV]=[CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName) for variableName in variableNames]
    return QV

def GetLagrangianVariable(t,
            varName='QCQI'):    
    variableNames = [varName]
    dataDictionary = MakeDataDictionary(variableNames,t)
    QCQI, = (dataDictionary[k] for k in variableNames)
    return QCQI

In [ ]:
#Calculation Functions
def CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, A1,A2, A1_Prior,A2_Prior,
                         QV,QV_Prior,QV_Prior_Eulerian,A2_Prior_Eulerian,
                         QCQI,QCQI_Prior,
                         isRemoveCondOrEvapEvents=True):
    """
    Function to compute 3D entrainment and update result array based on provided inputs.
    
    Returns a 3D (t,z) array containing the sum of the D array representing entrained parcels, by 1, and detrained parcels, by -1.
    The finally array is then ordered by the appropiate index using the np.add.at function
    
    Parameters:
    - A: The (t,p) lagrangian binary array.
    - T: The (t,p) lagrangian time index array.
    - Z: The (t,p) Lagrangian z index array.
    - Y: The (t,p) Lagrangian y index array.
    - X: The (t,p) Lagrangian x index array.

    """
    #Calculation for Entrainment and Detrainment
    DMatrix_Entrainment = SubtractA(A2,A2_Prior)
    DMatrix_Detrainment = DMatrix_Entrainment.copy()

    # Update D for entrainment/detrainment
    DMatrix_Entrainment[DMatrix_Entrainment < 0] = 0
    DMatrix_Detrainment[DMatrix_Detrainment > 0] = 0

    # # Require entraining parcel to be outside cloud at previous timestep #*testing
    # # D_raw = SubtractA(A2, A2_Prior)  
    # # entrainRaw = D_raw > 0
    # # entrainAfterQCQI = entrainRaw & (QCQI_Prior <= 1e-6)
    # # print("entrainment raw:", np.sum(entrainRaw))
    # # print("entrainment kept:", np.sum(entrainAfterQCQI))
    # # print("entrainment removed:", np.sum(entrainRaw & (QCQI_Prior > 1e-6)))
    # DMatrix_Entrainment[QCQI_Prior > 1e-6] = 0
    # DMatrix_Detrainment[QCQI > 1e-6] = 0

    ##########
    #General <==> Cloudy Updraft-Transfer Entrainment
    ########## c to g AND g to c
    SMatrix_Entrainment = ((A1_Prior == 1) & (A2 == 1) & (A2_Prior == 0)).astype(np.int8) #A1==>A2 entrainment #Note: last conditional removes double-counting where A1 = A2 = 1)
    SMatrix_Detrainment = SMatrix_Entrainment.copy()
    ##########

    # Remove in-situ condensation/evaporation events
    if isRemoveCondOrEvapEvents:
        DMatrix_Entrainment=RemoveCondOrEvapEvents(DMatrix_Entrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
        DMatrix_Detrainment=RemoveCondOrEvapEvents(DMatrix_Detrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
        SMatrix_Entrainment=RemoveCondOrEvapEvents(SMatrix_Entrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
        SMatrix_Detrainment=RemoveCondOrEvapEvents(SMatrix_Detrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
    
    # Initialize time and vertical dimension arrays
    Nz = ModelData.Nzh; Ny = ModelData.Nyh; Nx = ModelData.Nxh
    
    # Initialize result array
    Entrainment = np.zeros((Nz, Ny, Nx),dtype="int32"); Detrainment = Entrainment.copy()
    TransferEntrainment = Entrainment.copy(); TransferDetrainment = Entrainment.copy()
    
    # Initializing dry/moist entrainment arrays
    Entrainment_Dry = Entrainment.copy(); Entrainment_Moist = Entrainment.copy()

    if t==0:
        return (Entrainment,Detrainment,TransferEntrainment,TransferDetrainment,
                Entrainment_Dry,Entrainment_Moist)
    else:
        # Use np.add.at to accumulate values in the result array
        np.add.at(Entrainment, (Z, Y, X), DMatrix_Entrainment)
        np.add.at(Detrainment, (Z_Prior, Y_Prior, X_Prior), DMatrix_Detrainment)
        np.add.at(TransferEntrainment, (Z, Y, X), SMatrix_Entrainment)
        np.add.at(TransferDetrainment, (Z_Prior, Y_Prior, X_Prior), SMatrix_Detrainment)

        # Calculating dry/moist entrainment
        qv_u=QV
        # qv_u = GetQV_Updraft(QV_Prior_Eulerian, #depreciated, not needed
        #                      Z,Y,X, 
        #                      A2_Prior_Eulerian,DMatrix_Entrainment)
        [isDry,isMoist]  = CompareQuAndQe(qv_e=QV_Prior,qv_u=qv_u)

        dryMask = DMatrix_Entrainment * isDry
        moistMask = DMatrix_Entrainment * isMoist
        np.add.at(Entrainment_Dry, (Z, Y, X), dryMask) #dry entrainment
        np.add.at(Entrainment_Moist, (Z, Y, X),  moistMask) #moist entrainment

        
        return (Entrainment,Detrainment,TransferEntrainment,TransferDetrainment,
                Entrainment_Dry,Entrainment_Moist,
                dryMask.astype(bool),moistMask.astype(bool))

def SubtractA(A,A_Prior):
    D = np.zeros_like(A,dtype=np.int8)
    D = A*1 - A_Prior*1
    return D

def RemoveCondOrEvapEvents(array,
                           Z,Y,X,Z_Prior,Y_Prior,X_Prior):
    ParcelMoved = (Z != Z_Prior) | (Y != Y_Prior) | (X != X_Prior)
    # ParcelMoved = (Z == Z_Prior) & ((Y != Y_Prior) | (X != X_Prior)) #*testing only lateral entrainment
    array[~ParcelMoved]=0
    return array

In [ ]:
def CompareQuAndQe(qv_e,qv_u):
    validMask = np.isfinite(qv_u)

    isDry = (qv_e < qv_u) & validMask
    isMoist = (qv_e > qv_u) & validMask
    return isDry,isMoist

from scipy.spatial import cKDTree
def GetQV_Updraft(QV_Prior_Eulerian, #depreciated, not needed
                  Z,Y,X, 
                  A_Prior_Eulerian,DMatrix_Entrainment):
    qv_u = np.full(DMatrix_Entrainment.shape, np.nan, dtype=float)

    entrainParcelMask = DMatrix_Entrainment > 0
    if not np.any(entrainParcelMask): return qv_u

    cloudyGridMask = A_Prior_Eulerian == 1
    cloudy_pos = np.column_stack(np.where(cloudyGridMask))
    if cloudy_pos.shape[0] == 0: return qv_u

    entrain_pos = np.column_stack([Z[entrainParcelMask],
                                   Y[entrainParcelMask],X[entrainParcelMask]])

    tree = cKDTree(cloudy_pos)
    _, nearest = tree.query(entrain_pos, k=1)

    nearest_pos = cloudy_pos[nearest]

    qv_u[entrainParcelMask] = QV_Prior_Eulerian[nearest_pos[:, 0],
                                                nearest_pos[:, 1],
                                                nearest_pos[:, 2]]

    return qv_u

In [ ]:
#Running Function
def RunCalculation(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, A_g,A_c, A_g_Prior,A_c_Prior,
                   QV,QV_Prior,QV_Prior_Eulerian,
                   A_g_Prior_Eulerian,A_c_Prior_Eulerian,
                   QCQI,QCQI_Prior,
                   Processed_string,isRemoveCondOrEvapEvents=True): 

    [Entrainment_g, Detrainment_g,
     TransferEntrainment_g,TransferDetrainment_c,
     Entrainment_Dry_g,Entrainment_Moist_g,
     dryMask_g,moistMask_g] = CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior,
                                                                   A1=A_c,A2=A_g,
                                                                   A1_Prior=A_c_Prior,A2_Prior=A_g_Prior,
                                                                   QV=QV,QV_Prior=QV_Prior,
                                                                   QV_Prior_Eulerian=QV_Prior_Eulerian,
                                                                   A2_Prior_Eulerian=A_g_Prior_Eulerian,
                                                                   QCQI=QCQI,QCQI_Prior=QCQI_Prior,
                                                                   isRemoveCondOrEvapEvents\
                                                                   =isRemoveCondOrEvapEvents)

    [Entrainment_c, Detrainment_c,
     TransferEntrainment_c,TransferDetrainment_g,
     Entrainment_Dry_c,Entrainment_Moist_c,
     dryMask_c,moistMask_c] = CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, 
                                                                   A1=A_g,A2=A_c, 
                                                                   A1_Prior=A_g_Prior,A2_Prior=A_c_Prior,
                                                                   QV=QV,QV_Prior=QV_Prior,
                                                                   QV_Prior_Eulerian=QV_Prior_Eulerian,
                                                                   A2_Prior_Eulerian=A_c_Prior_Eulerian,
                                                                   QCQI=QCQI,QCQI_Prior=QCQI_Prior,
                                                                   isRemoveCondOrEvapEvents\
                                                                   =isRemoveCondOrEvapEvents)

    outputDictionary_Entrainment = {
        f"{Processed_string}Entrainment_g": Entrainment_g,
        f"{Processed_string}Entrainment_c": Entrainment_c,
        f"{Processed_string}TransferEntrainment_g": TransferEntrainment_g, #c to g
        f"{Processed_string}TransferEntrainment_c": TransferEntrainment_c, #g to c
        
        f"{Processed_string}Entrainment_Dry_g": Entrainment_Dry_g,
        f"{Processed_string}Entrainment_Moist_g": Entrainment_Moist_g,
        f"{Processed_string}Entrainment_Dry_c": Entrainment_Dry_c,
        f"{Processed_string}Entrainment_Moist_c": Entrainment_Moist_c,
        }
    
    outputDictionary_Detrainment = {
        f"{Processed_string}Detrainment_g": Detrainment_g,
        f"{Processed_string}Detrainment_c": Detrainment_c,
        f"{Processed_string}TransferDetrainment_g": TransferDetrainment_g, #from g to c
        f"{Processed_string}TransferDetrainment_c": TransferDetrainment_c, #from c to g
        }

    outputDictionary_Entrainment.update({
        f"{Processed_string}dryMask_g": dryMask_g,
        f"{Processed_string}moistMask_g": moistMask_g,
        f"{Processed_string}dryMask_c": dryMask_c,
        f"{Processed_string}moistMask_c": moistMask_c,
        })

    return outputDictionary_Entrainment, outputDictionary_Detrainment

In [ ]:
####################################
#CALCULATING ENTRAINMENT CONSTANT

In [ ]:
#Functions

#constants
def GetConstants():
    Cp=1004 #Jkg-1K-1
    Cv=717 #Jkg-1K-1
    Rd=Cp-Cv #Jkg-1K-1
    eps=0.608
    return Cp,Cv,Rd,eps

def GetNumerics():
    Np=ModelData.Np #number of lagrangian parcles

    # Lx=(ModelData.xf[-1]-ModelData.xf[0])*1000 #x length (m)
    # Ly=(ModelData.yf[-1]-ModelData.yf[0])*1000 #y length (m)
    dt=ModelData.dt #seconds
    dz=ModelData.dz #meters
    dy=ModelData.dy #meters
    dx=ModelData.dx #meters
    
    # zfs=ModelData.zf*1000
    # dz = np.diff(zfs)

    return Np, dt, dz,dy,dx

def zf(k):
    out=ModelData.zf[k]*1000
    return out

#calculation functions
# def rho(x,y,z,t):
#     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     p0=101325 #Pa
#     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     T=theta*(p/p0)**(Rd/Cp)
#     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     # Tv=T*(1+eps*qv)
#     Tv=T*(eps+qv)/(eps*(1+qv))
#     rho = p/(Rd*Tv)
#     out=rho
#     return out
    
def rho(x,y,z,rho_data_t):
    out=rho_data_t[z,y,x]
    return out
    
def Calculate_dm(t, dy,dx, Np):
    timeString = ModelData.timeStrings[t]
    rho_data_t = CallVariable(ModelData, DataManager, timeString, "rho")
    
    #calculating
    m=0
    for k in range(ModelData.Nzh):
        dz=(zf(k+1)-zf(k))
        for j in range(ModelData.Nyh):
            for i in range(ModelData.Nxh):
                rho_out=rho(i,j,k,rho_data_t)
                m+=rho_out*dz

    #multiplying by integration differentials
    dm = m*dx*dy/Np
    return dm

#calculating constant
def ComputeEntrainmentConstant(t=0):
    #constants
    [Cp,Cv,Rd,eps] = GetConstants()
    [Np, dt, dz,dy,dx] = GetNumerics()

    #calculation
    dm = Calculate_dm(t, dy,dx, Np); #print(dm)
    divisor=dt*dz*dy*dx
    entrainmentConstant = dm/divisor

    outputDictionary = {"entrainmentConstant": entrainmentConstant}
    return outputDictionary

#calculating constant
def ComputeMassFluxConstant(t=0):
    #constants
    # [Cp,Cv,Rd,eps] = GetConstants()
    [Np, dt, dz,dy,dx] = GetNumerics()

    #calculation
    dm = Calculate_dm(t, dy,dx, Np); print(dm)
    divisor=dz*dy*dx
    massFluxConstant = dm/divisor

    outputDictionary = {"massFluxConstant": massFluxConstant}
    return outputDictionary

In [ ]:
#Calculating
try:
    DataManager_Entrainment = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation",dtype='int32',verbose=False)

    entrainmentConstant = DataManager_Entrainment.LoadCalculations(DataManager_Entrainment.outputDataDirectory,
                                                                   dataName="EntrainmentConstant",
                                                                   verbose=False,)["entrainmentConstant"]
except:
    #calculating
    outputDictionary_EntrainmentConstant = ComputeEntrainmentConstant()
    entrainmentConstant = outputDictionary_EntrainmentConstant["entrainmentConstant"]
    
    #saving
    DataManager.SaveCalculations(DataManager.outputDataDirectory, outputDictionary_EntrainmentConstant, dataName="EntrainmentConstant",dtype="float32")

    #plotting
    plt.plot(entrainmentConstant,ModelData.zh,color='black')
    plt.ylabel("z (km)")
    plt.xlabel("Entrainment Constant (kg/m^3/s)")
    plt.title("Plotting Vertical Profile of Entrainment Constant\n (due to stretched z-grid)");

# def ApplyEntrainmentConstant(outputDictionary_Entrainment, outputDictionary_Detrainment):

#     for key, value in outputDictionary_Entrainment.items():
#         outputDictionary_Entrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]

#     for key, value in outputDictionary_Detrainment.items():
#         outputDictionary_Detrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]

def ApplyEntrainmentConstant(outputDictionary_Entrainment, outputDictionary_Detrainment):
    for key, value in outputDictionary_Entrainment.items():
        if "dryMask" in key or "moistMask" in key:
            continue
        outputDictionary_Entrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]
    for key, value in outputDictionary_Detrainment.items():
        if "dryMask" in key or "moistMask" in key:
            continue
        outputDictionary_Detrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]

In [ ]:
####################################
#CALCULATING

In [ ]:
t=np.abs(ModelData.time_hrs-15.5).argmin()
print(f'{ModelData.time_hrs[t]} LT')

#loading data
[Z,Y,X] = GetSpatialData(t); [Z_Prior,Y_Prior,X_Prior] = GetSpatialData(t-1)
[A_g,A_c] = GetAData(t); [A_g_Prior,A_c_Prior] = GetAData(t-1) 
[A_g_Prior_Eulerian,A_c_Prior_Eulerian]=GetAData_Eulerian(t-1)
QCQI = GetLagrangianVariable(t); QCQI_Prior = GetLagrangianVariable(t-1)

#loading moist/dry entrainment calculation variables
QV = GetVariableData(t); QV_Prior = GetVariableData(t-1)
# QV=QV+QCQI; QV_Prior=QV_Prior+QCQI_Prior#*testing
QV_Prior_Eulerian=GetVariableData_Eulerian(t-1)

#calculating variables
[outputDictionary_Entrainment,outputDictionary_Detrainment] = RunCalculation(t,
                                                                             Z,Y,X,Z_Prior,Y_Prior,X_Prior, A_g,A_c,
                                                                             A_g_Prior,A_c_Prior,
                                                                             QV,QV_Prior,QV_Prior_Eulerian,
                                                                             A_g_Prior_Eulerian,A_c_Prior_Eulerian,
                                                                             QCQI,QCQI_Prior,
                                                                             Processed_string)

#applying entrainment (density rate) constant
ApplyEntrainmentConstant(outputDictionary_Entrainment, outputDictionary_Detrainment)

#Getting Masks
dryMask = outputDictionary_Entrainment[f'{Processed_string}dryMask_c']
moistMask = outputDictionary_Entrainment[f'{Processed_string}moistMask_c']

In [ ]:
#calculating temperature
a = ModelData.OpenData()
prs = a['prs'].isel(time=t).data.compute(); prs_Prior = a['prs'].isel(time=t-1).data.compute()
Z = GetLagrangianVariable(t,varName='Z'); Z_Prior = GetLagrangianVariable(t-1,varName='Z')
Y = GetLagrangianVariable(t,varName='Y'); Y_Prior = GetLagrangianVariable(t-1,varName='Y')
X = GetLagrangianVariable(t,varName='X'); X_Prior = GetLagrangianVariable(t-1,varName='X')
TH = GetLagrangianVariable(t,varName='THETA'); TH_Prior = GetLagrangianVariable(t-1,varName='THETA') #replacement
PRS = prs[Z,Y,X]; PRS_Prior = prs_Prior[Z_Prior,Y_Prior,X_Prior]
T = CalculateT(PRS,TH); T_Prior = CalculateT(PRS_Prior,TH_Prior)

In [ ]:
####################################
#EXPLAINING WHERE "MOIST" ENTRAINMENT IS COMING FROM 

In [ ]:
#Calculation Functions
def GetCaseStudyIndex(mask,specificIndex):
    index = np.where(mask)[0][specificIndex] # pick one specific event

    # index = GetWarmerCase(moistMask,T,T_Prior,specificIndex=1) #Looing for warmer case instead
    return index
def GetCaseStudyIndex_WarmerCase(mask,specificIndex,
                                 T,T_Prior):
    warmingMask = mask & (T < T_Prior)
    index=np.where(warmingMask==True)[0][specificIndex]
    return index
def GetCaseStudyIndex_CoolerCase(mask,specificIndex,
                                 T,T_Prior):
    warmingMask = mask & (T > T_Prior)
    index=np.where(warmingMask==True)[0][specificIndex]
    return index

def GetVariables(timeStep,index,varName='qv',calculateZYX=True):        
    data = CallVariable(ModelData,DataManager,ModelData.timeStrings[timeStep],varName)
    if not calculateZYX:
        return [data]
    else:
        Z = GetLagrangianVariable(timeStep,varName='Z')[index]
        Y = GetLagrangianVariable(timeStep,varName='Y')[index]
        X = GetLagrangianVariable(timeStep,varName='X')[index]
        return [data,Z,Y,X]

def MakeBackTimeList(t, numberBack, stepSize=1):
    timeStepList = np.arange(t - numberBack*stepSize, t + stepSize, stepSize)
    return [timeStepList]

In [ ]:
#Plotting Functions
def MakePlot(array, X, Y, Z, timeStep, multiplier=1, varNameLabel=None, units=None,
             yLimit=(0,10),
             xWindow=250, yWindow=None, useLocalColorRange=False,
             cmap='turbo', symmetric=False,
             scatterColor='red',
             colorLimits=None,
             contourOverlayData=None,
             axis=None):
    arraySlice = array[:, Y, :] * multiplier
    varLabel = f'{varNameLabel} ({units})' if units else varNameLabel
    xCoord = ModelData.xh - ModelData.xh[0]
    yCoord = ModelData.zh
    xCenter = xCoord[X]
    xLo, xHi = xCenter - xWindow/2, xCenter + xWindow/2
    if yWindow is not None:
        yCenter = yCoord[Z]
        yLimit = (yCenter - yWindow/2, yCenter + yWindow/2)
    if colorLimits is not None:
        vmin, vmax = colorLimits
    elif useLocalColorRange:
        xMask = (xCoord >= xLo) & (xCoord <= xHi)
        yMask = (yCoord >= yLimit[0]) & (yCoord <= yLimit[1])
        visibleData = arraySlice[np.ix_(yMask, xMask)]
        vmin, vmax = np.nanmin(visibleData), np.nanmax(visibleData)
    else:
        vmin, vmax = np.nanmin(arraySlice), np.nanmax(arraySlice)
    # If symmetric requested (and cmap is diverging), force vmin = -vmax
    if symmetric:
        vabs = max(abs(vmin), abs(vmax))
        vmin, vmax = -vabs, vabs
    levels = np.linspace(vmin, vmax, 31)
    if axis is None:
        fig, axis = plt.subplots(figsize=(8,5))
    else:
        fig = axis.figure
    if cmap == 'RdBu_r' and vmin < 0 < vmax:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        contour = axis.contourf(xCoord, yCoord, arraySlice, levels=levels, cmap=cmap,
                                 norm=norm, extend='both')
    else:
        contour = axis.contourf(xCoord, yCoord, arraySlice, levels=levels, cmap=cmap, extend='both')
    if contourOverlayData is not None:
        axis.contour(xCoord, yCoord, contourOverlayData[:, Y, :], levels=[0.5], colors='white', linewidths=1.5,
                    zorder=2)

    colorbarTicks = np.linspace(vmin, vmax, 6) #colorbarTicks=None
    plt.colorbar(contour, ax=axis, label=varLabel, ticks=colorbarTicks)
    
    axis.plot(xCoord[X], yCoord[Z], marker='o', color=scatterColor, markersize=4,
              markeredgecolor=scatterColor,
              label=(f'Entrained Parcel\n'
                     f'{varNameLabel} = {array[Z,Y,X]*multiplier:.3f} ({units})\n'
                     f'z = {ModelData.zh[Z]*1e3:.1f} m'))
    axis.set_xlabel('X (km)')
    axis.set_ylabel('Z (km)')
    axis.set_title(f'{varNameLabel} cross-section\nt={ModelData.time_hrs[timeStep]} LT, y={ModelData.yh[Y]:.1f} km')
    axis.legend()
    axis.set_ylim(yLimit); axis.set_xlim(xLo, xHi)
    plt.tight_layout()
    colorLimits = (vmin, vmax)
    return axis,colorLimits

In [ ]:
#Calculating Case Study Index

index = GetCaseStudyIndex_WarmerCase(mask=moistMask,specificIndex=1,
                                     T=T,T_Prior=T_Prior) #Moist/Warm case (sim4: 3266)
index = GetCaseStudyIndex_CoolerCase(mask=moistMask,specificIndex=2,
                                     T=T,T_Prior=T_Prior) #Moist/Cool case (sim4: 3219434)

index = GetCaseStudyIndex_CoolerCase(mask=dryMask,specificIndex=1,
                                     T=T,T_Prior=T_Prior) #Dry/Cool case (sim4: 26822)
# index = GetCaseStudyIndex_WarmerCase(mask=dryMask,specificIndex=0,
#                                      T=T,T_Prior=T_Prior) #Dry/Warm case (sim4: 15517)

print(index)

In [ ]:
#QV cross-sections

#Variable Info
varName='qv'; varNameLabel=fr'$q_v$'
multiplier=1e3; units='g/kg'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=250; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='turbo', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='black',
                                    axis=axis)

In [ ]:
#QV cross-sections

#Variable Info
varName='qv'; varNameLabel=fr'$q_v$'
multiplier=1e3; units='g/kg'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=20; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='turbo', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='black',
                                    axis=axis)

In [ ]:
#QC cross-sections

#Variable Info
varName='qc'; varNameLabel=fr'$q_c$'
multiplier=1e3; units='g/kg'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=20; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='turbo', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='red',
                                    axis=axis)

In [ ]:
#QI cross-sections

#Variable Info
varName='qi'; varNameLabel=fr'$q_i$'
multiplier=1e3; units='g/kg'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=20; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='turbo', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='red',
                                    axis=axis)

In [ ]:
#W cross-sections

#Variable Info
varName='winterp'; varNameLabel=fr'$w$'
multiplier=1; units='m/s'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=20; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    symmetric=True,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='RdBu_r', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='black',
                                    axis=axis)

In [ ]:
#T cross-sections

#Variable Info
varName='T'; varNameLabel=fr'$T$'
multiplier=1; units='K'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=20; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='turbo', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='black',
                                    axis=axis)

In [ ]:
#RH cross-sections

#Variable Info
varName='RH_vapor'; varNameLabel=fr'$RH_v$'
multiplier=1e2; units='%'

#Times and X-Window
[timeStepList] = MakeBackTimeList(t, numberBack=3, stepSize=1)
xWindow=20; yLimit=(0,10)

#Plotting
numColumns = 2
numRows = int(np.ceil(len(timeStepList) / numColumns))
fig, axes = plt.subplots(numRows, numColumns, figsize=(8*numColumns, 5*numRows))
axes = np.array(axes).reshape(-1)  # flatten to 1D so zip still works regardless of grid shape

# Reverse both lists so t lands on the last axis, and we plot backward from there
timeStepListReversed = timeStepList[::-1]
axesReversed = axes[:len(timeStepListReversed)][::-1]

colorLimits = None
for axis, timeStep in tqdm(zip(axesReversed, timeStepListReversed), total=len(timeStepListReversed)):
    [data, Z, Y, X] = GetVariables(timeStep=timeStep, index=index, varName=varName)
    [axis, colorLimits] = MakePlot(data, X, Y, Z, timeStep, multiplier=multiplier,
                                    varNameLabel=varNameLabel, units=units,
                                    xWindow=xWindow, yLimit=yLimit,
                                    cmap='turbo', colorLimits=colorLimits, useLocalColorRange=True,
                                    scatterColor='black',
                                    axis=axis)

In [ ]:
####################################
#EXPLAINING HOW MUCH "MOIST" ENTRAINMENT IS OCCURING
#==> alot of both dry and moist entrainment occuring. mostly coming from below point of entrainment.

In [ ]:
#CALCULATIONS
####################################

In [ ]:
#Calculations
def CalculateT(P,TH):
    Rd_Cp = 287.0 / 1004.0   
    P0 = 1e5   
    
    T = TH * (P / P0) ** Rd_Cp
    return T

In [ ]:
#loading more variables
QV = GetLagrangianVariable(t,varName='QV'); QV_Prior = GetLagrangianVariable(t-1,varName='QV')

W = GetLagrangianVariable(t,varName='W'); W_Prior = GetLagrangianVariable(t-1,varName='W')
a=ModelData.OpenParcel(); W=a['w'].isel(time=t); W_Prior=a['w'].isel(time=t-1)

try:
    QC = GetLagrangianVariable(t,varName='QC'); QC_Prior = GetLagrangianVariable(t-1,varName='QC')
except:
    QC = GetLagrangianVariable(t,varName='QCQI'); QC_Prior = GetLagrangianVariable(t-1,varName='QCQI')
RH = GetLagrangianVariable(t,varName='RH_vapor'); RH_Prior = GetLagrangianVariable(t-1,varName='RH_vapor')

Z = GetLagrangianVariable(t,varName='Z'); Z_Prior = GetLagrangianVariable(t-1,varName='Z')

In [ ]:
#calculating rain
qr = a['qr'].isel(time=t).data.compute(); qr_Prior = a['qr'].isel(time=t-1).data.compute()
QR=np.zeros_like(Z, dtype='float32'); QR_Prior = QR.copy()
QR=qr[Z,Y,X];QR_Prior=qr_Prior[Z,Y,X]

In [ ]:
def GetData(mask): #mask=dryMask/moistMask
    qv_t=QV[mask]*1e3
    qv_tm1=QV_Prior[mask]*1e3
    
    qc_t=QC[mask]*1e3
    qc_tm1=QC_Prior[mask]*1e3

    qcqi_t=QCQI[mask]*1e3
    qcqi_tm1=QCQI_Prior[mask]*1e3
    
    rh_t=RH[mask]*1e2
    rh_tm1=RH_Prior[mask]*1e2

    t_t=T[mask]
    t_tm1=T_Prior[mask]

    th_t=TH[mask]
    th_tm1=TH_Prior[mask]
    
    w_t=W[mask]
    w_tm1=W_Prior[mask]

    qr_t=QR[mask]*1e3
    qr_tm1=QR_Prior[mask]*1e3

    z_t=ModelData.zh[Z[mask]]*1e3
    z_tm1=ModelData.zh[Z_Prior[mask]]*1e3

    return [qv_t,qv_tm1,
            qc_t,qc_tm1,
            qcqi_t,qcqi_tm1,
            rh_t,rh_tm1,
            t_t,t_tm1,
            th_t,th_tm1,
            w_t,w_tm1,
            qr_t,qr_tm1,
            z_t,z_tm1]
    
[qv_t_D,qv_tm1_D,
 qc_t_D,qc_tm1_D,
 qcqi_t_D,qcqi_tm1_D,
 rh_t_D,rh_tm1_D,
 t_t_D,t_tm1_D,
 th_t_D,th_tm1_D,
 w_t_D,w_tm1_D,
 qr_t_D,qr_tm1_D,
 z_t_D,z_tm1_D] = GetData(mask=dryMask)

[qv_t_M,qv_tm1_M,
 qc_t_M,qc_tm1_M,
 qcqi_t_M,qcqi_tm1_M,
 rh_t_M,rh_tm1_M,
 t_t_M,t_tm1_M,
 th_t_M,th_tm1_M,
 w_t_M,w_tm1_M,
 qr_t_M,qr_tm1_M,
 z_t_M,z_tm1_M] = GetData(mask=moistMask)

In [ ]:
#PLOTTING FUNCTIONS
####################################

In [ ]:
def PrepareHistData(data1, data2, loPercentile=5, hiPercentile=95, useWeights=True):
    """Compute normalized weights (optional) and shared x-limits for a dry/moist histogram pair."""
    if useWeights:
        totalN = len(data1) + len(data2)
        weights1 = np.ones_like(data1) / totalN * 100
        weights2 = np.ones_like(data2) / totalN * 100
    else:
        weights1 = None
        weights2 = None

    xlo = min(np.percentile(data1, loPercentile), np.percentile(data2, loPercentile))
    xhi = max(np.percentile(data1, hiPercentile), np.percentile(data2, hiPercentile))
    return [weights1, weights2, xlo, xhi]


def PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi, xLabel, titleText,
                      showMedian=False, useSciNotation=False):
    """Plot side-by-side Dry/Moist histograms sharing y-axis and x-limits. Weights are optional."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

    panelData = [data1, data2]
    panelWeights = [weights1, weights2]
    panelTitles = ["Dry", "Moist"]

    for axis, dataSet, weightSet, panelTitle in zip(axes, panelData, panelWeights, panelTitles):
        if weightSet is not None:
            axis.hist(dataSet, bins=100, range=(xlo, xhi), weights=weightSet)
        else:
            axis.hist(dataSet, bins=100, range=(xlo, xhi))
        if showMedian:
            axis.axvline(np.median(dataSet), color='black', zorder=1)
        axis.set_xlim(xlo, xhi)
        axis.set_title(panelTitle)

    for axis in axes:
        axis.set_xlabel(xLabel)

    plt.suptitle(titleText)
    axes[0].set_ylabel('% / (dry + moist)' if weights1 is not None else 'Count')
    plt.tight_layout()

    if useSciNotation:
        apply_scientific_notation(axes.ravel())

    return [fig, axes]

In [ ]:
#PLOTTING
####################################

In [ ]:
#Plotting
#### ============================================================
# QV_DIFF (weighted, as before)
# ============================================================
data1 = qv_tm1_D - qv_t_D
data2 = qv_tm1_M - qv_t_M
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=1, hiPercentile=99)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'qv (g/kg)', fr'$\phi_e - \phi_u$', showMedian=True)

# ============================================================
# T_DIFF
# ============================================================
data1 = t_tm1_D - t_t_D
data2 = t_tm1_M - t_t_M
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=1, hiPercentile=99)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'T (K)', fr'$\phi_e - \phi_u$', showMedian=True)



# ============================================================
# QC_ENV (example with weights turned off)
# ============================================================
data1 = qc_tm1_D; data2 = qc_tm1_M
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=5, hiPercentile=95,
                                                  useWeights=False)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'qc (g/kg)', fr'$\phi_e$', showMedian=False, useSciNotation=True)

# ============================================================
# RH_ENV
# ============================================================
data1 = rh_tm1_D; data2 = rh_tm1_M
data1 = data1[(data1 >= 100)]; data2 = data2[(data2 >= 100)]
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=5, hiPercentile=95)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'rh (%) >= 100', fr'$\phi_e$', showMedian=True)


# ============================================================
# QR_ENV
# ============================================================
data1 = qr_tm1_D; data2 = qr_tm1_M
# data1=data1[(data1<=1e-3)]; data2=data2[(data2<=1e-3)]
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=5, hiPercentile=95)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'qr (g/kg)', fr'$\phi_e$', showMedian=False)


# ============================================================
# W_DIFF
# ============================================================
data1 = w_tm1_D - w_t_D
data2 = w_tm1_M - w_t_M
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=1, hiPercentile=99)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'w (m/s)', fr'$\phi_e - \phi_u$', showMedian=True)
# qc_tm1_M[qv_tm1_M>qv_t_M].max()>1e-3 are any moist entrainment cases in cloud prior to entrainment ==> no


# ============================================================
# Z_DIFF
# ============================================================
data1 = z_tm1_D - z_t_D
data2 = z_tm1_M - z_t_M
[weights1, weights2, xlo, xhi] = PrepareHistData(data1, data2, loPercentile=1, hiPercentile=99)
[fig, axes] = PlotDryMoistHist(data1, data2, weights1, weights2, xlo, xhi,
                                'z_grid (m)', fr'$\phi_e - \phi_u$', showMedian=True)
# qc_tm1_M[qv_tm1_M>qv_t_M].max()>1e-3 are any moist entrainment cases in cloud prior to entrainment ==> no

In [ ]:
####################################
#MORE TESTING

In [ ]:
# #Plotting Dry and Moist Entrainment

# def TestPlot_Average(e,e_dry,e_moist,updraftType,
#              cmap='turbo',numLevels=101):

#     numLevels=100
    
#     fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
    
#     plotX = ModelData.xh-ModelData.xh[0]
#     plotY = ModelData.zh
    
#     data = [(e,       'Total'),
#             (e_dry,   'Dry'),
#             (e_moist, 'Moist'),]
    
#     # shared color scale across all three panels so they're visually comparable
#     vmax = max(np.nanmax(np.mean(d, axis=1)) for d, _ in data); vmin = min(np.nanmin(np.mean(d, axis=1)) for d, _ in data)
#     levels = np.linspace(vmin, vmax, numLevels)
    
#     for ax, (d, title) in zip(axes, data):
#         plotZ = np.mean(d, axis=1)
#         cf = ax.contourf(plotX, plotY, plotZ, levels=levels, cmap=cmap)
#         ax.set_title(title);

#         yLims=(0,1.5) if updraftType=='g' else (1.5,4)
#         xLims=(150,None)
        
#         ax.set_ylim(yLims)
#         ax.set_xlim(xLims)
#         ax.set_xlabel('x (km)')
    
#     axes[0].set_ylabel('z (km)')
    
#     cbar=fig.colorbar(cf, ax=axes, shrink=0.8, label='Entrainment')
#     apply_scientific_notation_colorbar([cbar])


# updraftType='g'
# updraftType='c'

# e=outputDictionary_Entrainment[f'{Processed_string}Entrainment_{updraftType}']
# e_dry=outputDictionary_Entrainment[f'{Processed_string}Entrainment_Dry_{updraftType}']
# e_moist=outputDictionary_Entrainment[f'{Processed_string}Entrainment_Moist_{updraftType}']
# TestPlot_Average(e,e_dry,e_moist,updraftType)

In [ ]:
# #Condensation Check

# timeStep=t
# [a,Z,Y,X]=GetVariables(timeStep=timeStep,index=index,varName='qv')
# [b]=GetVariables(timeStep=timeStep,index=index,varName='qc',calculateZYX=False)
# [c]=GetVariables(timeStep=timeStep,index=index,varName='qi',calculateZYX=False)

# print('qv',f'{a[Z,Y,X]*1e3:.3f}',' g/kg')
# print('qc+qi',f'{(b[Z,Y,X]+c[Z,Y,X])*1e3:.3f}',' g/kg')
# print('qv+qc+qi',f'{(a[Z,Y,X]+b[Z,Y,X]+c[Z,Y,X])*1e3:.3f}',' g/kg')

# timeStep=t-1
# [a,Z,Y,X]=GetVariables(timeStep=timeStep,index=index,varName='qv')
# [b]=GetVariables(timeStep=timeStep,index=index,varName='qc',calculateZYX=False)
# [c]=GetVariables(timeStep=timeStep,index=index,varName='qi',calculateZYX=False)

# print('qv',f'{a[Z,Y,X]*1e3:.3f}',' g/kg')
# print('qc+qi',f'{(b[Z,Y,X]+c[Z,Y,X])*1e3:.3f}',' g/kg')
# print('qv+qc+qi',f'{(a[Z,Y,X]+b[Z,Y,X]+c[Z,Y,X])*1e3:.3f}',' g/kg')

In [ ]:
# #More investigation into Hydrometeors and air holding more moisture

# #full hydrometeor conserved changed check
# a = ModelData.OpenData()
# qv = a['qv'].isel(time=t).data.compute(); qv_Prior = a['qv'].isel(time=t-1).data.compute()
# qc = a['qc'].isel(time=t).data.compute(); qc_Prior = a['qc'].isel(time=t-1).data.compute()
# qi = a['qi'].isel(time=t).data.compute(); qi_Prior = a['qi'].isel(time=t-1).data.compute()

# Z = GetLagrangianVariable(t,varName='Z'); Z_Prior = GetLagrangianVariable(t-1,varName='Z')
# Y = GetLagrangianVariable(t,varName='Y'); Y_Prior = GetLagrangianVariable(t-1,varName='Y')
# X = GetLagrangianVariable(t,varName='X'); X_Prior = GetLagrangianVariable(t-1,varName='X')

# QV = qv[Z,Y,X]; QV_Prior = qv_Prior[Z_Prior,Y_Prior,X_Prior]
# QC = qc[Z,Y,X]; QC_Prior = qc_Prior[Z_Prior,Y_Prior,X_Prior]
# QI = qi[Z,Y,X]; QI_Prior = qi_Prior[Z_Prior,Y_Prior,X_Prior]

# qv_t_M=QV[moistMask]; qv_tm1_M=QV_Prior[moistMask]     # <-- rebuild these here too
# qc_t_M=QC[moistMask]; qc_tm1_M=QC_Prior[moistMask]
# qi_t_M=QI[moistMask]; qi_tm1_M=QI_Prior[moistMask]

# qt_t_M = qv_t_M + qc_t_M + qi_t_M
# qt_tm1_M = qv_tm1_M + qc_tm1_M + qi_tm1_M

In [ ]:
# #full hydrometeor conserved changed check
# a = ModelData.OpenData()
# qv = a['qv'].isel(time=t).data.compute(); qv_Prior = a['qv'].isel(time=t-1).data.compute()
# qc = a['qc'].isel(time=t).data.compute(); qc_Prior = a['qc'].isel(time=t-1).data.compute()
# qi = a['qi'].isel(time=t).data.compute(); qi_Prior = a['qi'].isel(time=t-1).data.compute()
# qr = a['qr'].isel(time=t).data.compute(); qr_Prior = a['qr'].isel(time=t-1).data.compute()
# qs = a['qs'].isel(time=t).data.compute(); qs_Prior = a['qs'].isel(time=t-1).data.compute()
# qg = a['qg'].isel(time=t).data.compute(); qg_Prior = a['qg'].isel(time=t-1).data.compute()

# Z = GetLagrangianVariable(t,varName='Z'); Z_Prior = GetLagrangianVariable(t-1,varName='Z')
# Y = GetLagrangianVariable(t,varName='Y'); Y_Prior = GetLagrangianVariable(t-1,varName='Y')
# X = GetLagrangianVariable(t,varName='X'); X_Prior = GetLagrangianVariable(t-1,varName='X')

# QV = qv[Z,Y,X]; QV_Prior = qv_Prior[Z_Prior,Y_Prior,X_Prior]
# QC = qc[Z,Y,X]; QC_Prior = qc_Prior[Z_Prior,Y_Prior,X_Prior]
# QI = qi[Z,Y,X]; QI_Prior = qi_Prior[Z_Prior,Y_Prior,X_Prior]
# QR = qr[Z,Y,X]; QR_Prior = qr_Prior[Z_Prior,Y_Prior,X_Prior]
# QS = qs[Z,Y,X]; QS_Prior = qs_Prior[Z_Prior,Y_Prior,X_Prior]
# QG = qg[Z,Y,X]; QG_Prior = qg_Prior[Z_Prior,Y_Prior,X_Prior]

# qv_t_M=QV[moistMask]; qv_tm1_M=QV_Prior[moistMask]
# qc_t_M=QC[moistMask]; qc_tm1_M=QC_Prior[moistMask]
# qi_t_M=QI[moistMask]; qi_tm1_M=QI_Prior[moistMask]
# qr_t_M=QR[moistMask]; qr_tm1_M=QR_Prior[moistMask]
# qs_t_M=QS[moistMask]; qs_tm1_M=QS_Prior[moistMask]
# qg_t_M=QG[moistMask]; qg_tm1_M=QG_Prior[moistMask]

# # Full total water content, including all hydrometeor species
# qt_t_M = qv_t_M + qc_t_M + qi_t_M + qr_t_M + qs_t_M + qg_t_M
# qt_tm1_M = qv_tm1_M + qc_tm1_M + qi_tm1_M + qr_tm1_M + qs_tm1_M + qg_tm1_M

# fracInconsistent = np.mean(qt_t_M > qt_tm1_M)
# print(f"Fraction of moist cases where full qt increased: {fracInconsistent*100:.2f}%")

In [ ]:
# # Split by parcel's own temperature change direction (t-1 vs t)
# coolingMask = t_tm1_M > t_t_M   # parcel cooled (T_prior warmer than T_current)
# warmingMask = t_tm1_M < t_t_M   # parcel warmed (T_prior cooler than T_current)

# qtIncreaseMask = qt_t_M > qt_tm1_M  # total water (qv+qc+qi) increased

# nCooling = np.sum(coolingMask)
# nWarming = np.sum(warmingMask)

# nCoolingAndQtIncrease = np.sum(coolingMask & qtIncreaseMask)
# nWarmingAndQtIncrease = np.sum(warmingMask & qtIncreaseMask)

# print(f"Total moist cases: {len(t_t_M)}")
# print()
# print(f"Cooling cases (t_tm1 > t_t): {nCooling} ({nCooling/len(t_t_M)*100:.1f}%)")
# print(f"  Of these, qt increased: {nCoolingAndQtIncrease} "
#       f"({nCoolingAndQtIncrease/nCooling*100:.1f}% of cooling cases)")
# print()
# print(f"Warming cases (t_tm1 < t_t): {nWarming} ({nWarming/len(t_t_M)*100:.1f}%)")
# print(f"  Of these, qt increased: {nWarmingAndQtIncrease} "
#       f"({nWarmingAndQtIncrease/nWarming*100:.1f}% of warming cases)")

In [ ]:
# # Compute saturation mixing ratio at both times for the moist subset

# def SaturationVaporPressure(T):
#     """
#     Bolton (1980) approximation of Clausius-Clapeyron Equation.
#     T in Kelvin, returns saturation vapor pressure in Pa.
#     """
#     TC = T - 273.15
#     EsHpa = 6.112 * np.exp((17.67 * TC) / (TC + 243.5))
#     return EsHpa * 100.0

# def SaturationMixingRatio(T, P, Epsilon=0.622):
#     """
#     Calculates saturation water vapor mixing ratio, kg/kg.
#     Derived by q_v = rho_v/rho_d = [e/(RvT)] / [(P-e)/(RdT)]
#                = (Rd/Rv)*[e/(P-e)] = epsilon*[e/(P-e)]

#     T: temperature, K
#     P: pressure, Pa
#     """
#     Es = SaturationVaporPressure(T)
#     return Epsilon * Es / (P - Es)
# qsat_t_M = SaturationMixingRatio(t_t_M, prs_t_M)      # q_sat at time t
# qsat_tm1_M = SaturationMixingRatio(t_tm1_M, prs_tm1_M)  # q_sat at time t-1
# prs_t_M=PRS[moistMask];prs_tm1_M=PRS_Prior[moistMask]

# # Check: is qsat at t less than qsat at t-1?
# qsatDecreaseMask = qsat_t_M < qsat_tm1_M

# nQsatDecrease = np.sum(qsatDecreaseMask)
# nTotal = len(qsat_t_M)

# print(f"Total moist cases: {nTotal}")
# print(f"Cases where qsat decreased (qsat_t < qsat_tm1): {nQsatDecrease} ({nQsatDecrease/nTotal*100:.1f}%)")